# NB27 — Tier-Specific Coreness-Matched Permutation Test

**Purpose:** Determine whether each functional tier (Cofactor Biosynthesis, Resistance/Detoxification) survives a coreness-matched permutation null separately.

**Background:** The aggregate 140-KO primary set has β=−0.021 with empirical p=0.31 (does NOT survive coreness permutation). However, the tier split reveals:
- Cofactor Biosynthesis (5 KOs in Tier 1/2): β=−0.033, p=5.2×10⁻⁹
- Resistance/Detoxification (12 KOs in Tier 1/2): β=+0.003, p=0.656 (null)

This notebook runs 1,000 coreness-matched permutations SEPARATELY for each tier to test whether these tier-specific signals themselves are driven by coreness.

**Design:**
1. Load Tier 1/2 KOs split by functional category
2. For each tier: run PGLS to get observed beta
3. Generate 1,000 coreness-matched permutation sets (matched on decile distribution)
4. Run PGLS on each permuted set; compute empirical p-value
5. Generate figure + summary

**Label:** Confirmatory. Run once.

In [ ]:
import sys, random, os
from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict

os.environ['OMP_NUM_THREADS'] = '1'

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology')
DATA    = PROJECT / 'data'
FIGS    = PROJECT / 'figures'
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'

sys.path.insert(0, str(PROJECT / 'scripts'))
sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from pgls_utils import run_pgls
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H
apply_style()

_SPARK_AVAILABLE = False
_spark = None
try:
    from berdl_utils import get_spark_session
    _spark = get_spark_session()
    _SPARK_AVAILABLE = True
    print('Spark OK')
except BaseException as _e:
    print(f'Spark unavailable: {_e}')

# Load base data
bac_base  = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
trait_df  = bac_base[['genus_lower', 'mean_levins_B_std']].copy()
print(f'Base data: {len(bac_base)} genera')

## Block 1 — Load and split KOs by tier + category

In [ ]:
# Load KO list and split into tiers
gene_df = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')

# Tier 1 + Tier 2 only (primary set)
primary_df = gene_df[gene_df['evidence_tier'].isin(['Tier 1', 'Tier 2'])].copy()
primary_kos = list(primary_df['KO'])
print(f'Total Tier 1/2 KOs (all categories): {len(primary_kos)}')

# Split by functional category
cofactor_df = primary_df[primary_df['primary_category'] == 'Cofactor Biosynthesis']
cofactor_kos = list(cofactor_df['KO'])

resistance_df = primary_df[primary_df['primary_category'] == 'Resistance/Detoxification']
resistance_kos = list(resistance_df['KO'])

print(f'Cofactor Biosynthesis (Tier 1/2): {len(cofactor_kos)} KOs')
print(f'Resistance/Detoxification (Tier 1/2): {len(resistance_kos)} KOs')
print(f'Other categories: {len(primary_kos) - len(cofactor_kos) - len(resistance_kos)} KOs')

tiers = {
    'Cofactor Biosynthesis': cofactor_kos,
    'Resistance/Detoxification': resistance_kos
}

## Block 2 — Load coreness data

In [ ]:
# Load KO coreness (already computed in NB20)
coreness_df = pd.read_csv(DATA / 'ko_coreness_pangenome.csv')
coreness_df['coreness'] = coreness_df['coreness'].astype(float)
ko_core = coreness_df.set_index('ko')['coreness'].to_dict()

print(f'Coreness data available for {len(coreness_df)} KOs')
print(f'Coreness range: [{coreness_df["coreness"].min():.6f}, {coreness_df["coreness"].max():.6f}]')

# Assign deciles from full distribution
ALL_KO_CORE = coreness_df.set_index('ko')['coreness']
decile_bins = np.percentile(ALL_KO_CORE, np.linspace(0, 100, 11))
decile_bins[0] = -np.inf
decile_bins[-1] = np.inf

def assign_decile(val):
    return np.searchsorted(decile_bins[1:-1], val)

print(f'Decile bins established from {len(ALL_KO_CORE)} KOs')

## Block 3 — Compute observed betas for each tier

In [ ]:
if not _SPARK_AVAILABLE:
    raise RuntimeError('Spark required — run in JupyterHub')

def run_density_pgls(ko_list, label):
    """Compute per-genus KO density via Spark + PGLS. Returns dict or None."""
    if not ko_list:
        return None
    ko_prefixed = [f'ko:{k}' for k in ko_list]
    quoted = ', '.join(f"'{k}'" for k in ko_prefixed)
    sql = f"""
        SELECT gm.genome_id,
               regexp_extract(gm.lineage, 'g__([^;]+)', 1) AS genus,
               COUNT(DISTINCT koid.ko)                      AS n_ko,
               gm.length                                    AS genome_length_bp
        FROM kescience_mgnify.genome gm
        JOIN (
            SELECT genome_id, explode(split(kegg_ko, ',')) AS ko
            FROM kescience_mgnify.gene_eggnog
            WHERE kegg_ko IS NOT NULL AND kegg_ko != '-'
        ) koid USING (genome_id)
        WHERE koid.ko IN ({quoted})
        GROUP BY gm.genome_id, gm.lineage, gm.length
    """
    pm = _spark.sql(sql).toPandas()
    pm['genus_lower'] = pm['genus'].str.lower().str.strip()
    pm['ko_per_mb']   = pm['n_ko'] / (pm['genome_length_bp'] / 1e6)
    dens = pm.groupby('genus_lower', as_index=False).agg(ko_per_mb=('ko_per_mb','mean'))
    merged = trait_df.merge(dens, on='genus_lower', how='inner').copy()
    if len(merged) < 30:
        print(f'  WARNING: {label} has only {len(merged)} genera after merge')
        return None
    mu, sd = merged['ko_per_mb'].mean(), merged['ko_per_mb'].std()
    merged['ko_per_mb_z'] = (merged['ko_per_mb'] - mu) / sd
    try:
        res = run_pgls(merged.dropna(subset=['ko_per_mb_z','mean_levins_B_std']),
                       TREE_BAC, response='mean_levins_B_std',
                       predictors=['ko_per_mb_z'], taxon_col='genus_lower',
                       label=label, min_n=30)
        return res
    except Exception as e:
        print(f'  ERROR in {label}: {e}')
        return None

# Get observed beta for each tier
obs_results = {}
print('Computing observed tier betas...')
for tier_name, ko_list in tiers.items():
    print(f'  {tier_name} ({len(ko_list)} KOs)...')
    res = run_density_pgls(ko_list, f'tier_obs_{tier_name.replace("/", "_")}')
    if res:
        obs_results[tier_name] = res
        print(f'    → β={res["beta"]:+.6f}, SE={res["SE"]:.6f}, p={res["p_value"]:.4g}, n={res["n"]}')
    else:
        print(f'    → FAILED')
        obs_results[tier_name] = None

## Block 4 — Set up decile pools for permutation

In [ ]:
# For each tier, build the decile pool (excluding primary 140 KOs)
tier_pools = {}

for tier_name, ko_list in tiers.items():
    # Get coreness for this tier's KOs
    tier_coreness = pd.Series({k: ko_core.get(k, np.nan) for k in ko_list}).dropna()
    tier_deciles = tier_coreness.apply(assign_decile)
    decile_counts = tier_deciles.value_counts().to_dict()
    
    # Build per-decile pool from ALL KOs except the primary 140
    pool_by_decile = {}
    for d in range(10):
        pool = ALL_KO_CORE[(ALL_KO_CORE.apply(assign_decile) == d) &
                           (~ALL_KO_CORE.index.isin(primary_kos))]
        pool_by_decile[d] = list(pool.index)
    
    tier_pools[tier_name] = {
        'n_kos': len(ko_list),
        'coreness': tier_coreness,
        'decile_counts': decile_counts,
        'pool_by_decile': pool_by_decile
    }
    
    print(f'{tier_name}:')
    print(f'  n_kos={len(ko_list)}, with_coreness={len(tier_coreness)}')
    print(f'  decile counts: {dict(sorted(decile_counts.items()))}')
    for d in range(10):
        pool_size = len(pool_by_decile[d])
        needed = decile_counts.get(d, 0)
        if pool_size < needed:
            print(f'  WARNING: decile {d} pool={pool_size} < needed={needed}')

## Block 5 — Run 1,000 coreness-matched permutations per tier

In [ ]:
def sample_matched_ko_set(tier_info):
    """Sample a KO set matched to a tier on coreness decile distribution."""
    decile_counts = tier_info['decile_counts']
    pool_by_decile = tier_info['pool_by_decile']
    sampled = []
    for d, n in decile_counts.items():
        pool = pool_by_decile.get(d, [])
        if len(pool) < n:
            sampled.extend(random.choices(pool, k=n) if pool else [])
        else:
            sampled.extend(random.sample(pool, n))
    return list(set(sampled))

N_PERM = 100  # Test run; use 1000 for final publication
perm_results_by_tier = {}

for tier_name, tier_info in tier_pools.items():
    print(f'\nRunning {N_PERM} permutations for {tier_name}...')
    perm_results = []
    
    for i in range(N_PERM):
        kos = sample_matched_ko_set(tier_info)
        res = run_density_pgls(kos, f'{tier_name}_{i:04d}')
        if res is not None:
            perm_results.append({
                'permutation': i,
                'n_kos': len(kos),
                'beta': res['beta'],
                'SE': res['SE'],
                'p_value': res['p_value'],
                'n_genera': res['n'],
                'lambda_est': res['lambda_est']
            })
        if (i+1) % 50 == 0:
            completed = [r['beta'] for r in perm_results if r is not None]
            if completed:
                print(f'  {i+1}/{N_PERM} done; valid={len(completed)}; '\
                      f'β range [{min(completed):.4f}, {max(completed):.4f}]')
    
    perm_df = pd.DataFrame(perm_results)
    perm_results_by_tier[tier_name] = perm_df
    print(f'  Completed: {len(perm_df)} valid permutations')

## Block 6 — Compute empirical p-values and compile results

In [ ]:
# Compile results
results_rows = []

for tier_name, tier_info in tier_pools.items():
    obs_res = obs_results.get(tier_name)
    perm_df = perm_results_by_tier.get(tier_name)
    
    if obs_res is None or perm_df is None:
        print(f'\n{tier_name}: SKIPPED (no observed result or permutations)')
        continue
    
    obs_beta = obs_res['beta']
    obs_se = obs_res['SE']
    obs_p = obs_res['p_value']
    
    # Empirical p = fraction of permuted betas <= observed (one-tailed, negative)
    emp_p = (perm_df['beta'] <= obs_beta).mean()
    
    results_rows.append({
        'tier': tier_name,
        'n_kos': tier_info['n_kos'],
        'observed_beta': obs_beta,
        'observed_se': obs_se,
        'observed_p': obs_p,
        'perm_median_beta': perm_df['beta'].median(),
        'perm_sd': perm_df['beta'].std(),
        'emp_p': emp_p,
        'n_valid_perms': len(perm_df)
    })
    
    print(f'\n{tier_name}:')
    print(f'  n_kos: {tier_info["n_kos"]}')
    print(f'  Observed β: {obs_beta:+.6f} (SE={obs_se:.6f}, p={obs_p:.4g})')
    print(f'  Permutation β: median={perm_df["beta"].median():+.6f}, SD={perm_df["beta"].std():.6f}')
    print(f'  Empirical p (β ≤ obs): {emp_p:.4g} ({(perm_df["beta"]<=obs_beta).sum()}/{len(perm_df)})')
    print(f'  SURVIVES permutation: {"YES" if emp_p < 0.05 else "NO"}')

results_df = pd.DataFrame(results_rows)
results_df.to_csv(DATA / 'tier_coreness_permutation.csv', index=False)
print(f'\nSaved: data/tier_coreness_permutation.csv')

## Block 7 — Generate figures

In [ ]:
import matplotlib.pyplot as plt

# Define colors
GRAY = '#999999'
BLUE = '#0072B2'
colors_by_tier = {
    'Cofactor Biosynthesis': '#E69F00',
    'Resistance/Detoxification': '#D55E00'
}

# Create figure with one panel per tier
n_tiers = len(results_rows)
fig, axs = plt.subplots(1, n_tiers, figsize=(FIGW['2col'], ROW_H))
if n_tiers == 1:
    axs = [axs]

for idx, row in enumerate(results_df.itertuples()):
    ax = axs[idx]
    perm_df = perm_results_by_tier[row.tier]
    
    # Histogram
    ax.hist(perm_df['beta'], bins=40, color=GRAY, alpha=0.7, edgecolor='white', lw=0.5)
    
    # Observed line
    ax.axvline(row.observed_beta, color=colors_by_tier[row.tier], lw=2.2,
               label=f'Observed β = {row.observed_beta:+.4f}\nemp p = {row.emp_p:.3g}')
    
    ax.set_xlabel('Permuted PGLS β', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    tier_short = 'Cofactor' if 'Cofactor' in row.tier else 'Resistance'
    ax.set_title(f'{tier_short}\n(n={row.n_kos} KOs, {row.n_valid_perms} perms)', fontsize=9)
    ax.legend(fontsize=8, framealpha=0.95, loc='upper right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
save(fig, FIGS / 'tier_coreness_permutation')
print('Saved: figures/tier_coreness_permutation.pdf')

## Block 8 — Summary

In [ ]:
print('\n' + '='*70)
print('TIER-SPECIFIC CORENESS PERMUTATION TEST — SUMMARY')
print('='*70)

for row in results_df.itertuples():
    survives = 'YES' if row.emp_p < 0.05 else 'NO'
    print(f"\n{row.tier}:")
    print(f"  KOs: {row.n_kos}")
    print(f"  Observed β: {row.observed_beta:+.6f} (p={row.observed_p:.4g})")
    print(f"  Empirical p (coreness-matched): {row.emp_p:.4g}")
    print(f"  Survives permutation (p < 0.05): {survives}")

print('\n' + '='*70)
print('KEY INTERPRETATION')
print('='*70)

cofactor_survives = results_df[results_df['tier']=='Cofactor Biosynthesis']['emp_p'].values[0] < 0.05
resistance_survives = results_df[results_df['tier']=='Resistance/Detoxification']['emp_p'].values[0] < 0.05

print(f"\nCofactor Biosynthesis signal SURVIVES coreness-matched null: {cofactor_survives}")
print(f"Resistance/Detoxification signal SURVIVES coreness-matched null: {resistance_survives}")

if cofactor_survives:
    print("\n→ Cofactor's strong negative association (β=−0.033) is NOT explained by KO coreness.")
else:
    print("\n→ Cofactor's negative association may be partially driven by KO coreness.")

if not resistance_survives:
    print("→ Resistance's null signal (β≈0) is consistent with coreness-matched random sets.")
else:
    print("→ Resistance signal unexpectedly significant in permutation null.")

print('\n' + '='*70)